In [76]:
# !python -c "import sys; print(sys.executable)"

# LangChain RAG


### Imports

In [77]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os

### Loading Documents

In [78]:
def load_documents(doc_path):
    """Load documents from the specified path."""

    if not os.path.exists(doc_path):
        raise FileNotFoundError(f"Document not found: {doc_path}")
    
    loader = DirectoryLoader(doc_path, 
                             glob="*.txt",
                             show_progress=True,
                             loader_cls=TextLoader,
                             loader_kwargs={"encoding": "utf-8"})
    
    documents = loader.load()

    if len(documents) == 0:
        raise ValueError(f"No .txt files found in the specified path: {doc_path}")
    
    return documents

In [79]:
documents = load_documents(r"D:\Projects\Agentic-AI\langchain-RAG\docs")


100%|██████████| 5/5 [00:00<00:00, 49.87it/s]


In [80]:
documents[0].__dict__

{'id': None,
 'metadata': {'source': 'D:\\Projects\\Agentic-AI\\langchain-RAG\\docs\\Google.txt'},
 'page_content': '\ufeffGoogle\nGoogle LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC\nAmerican multinational corporation and technology\ncompany focusing on online advertising, search engine\ntechnology, cloud computing, computer software,\nquantum computing, e-commerce, consumer\nelectronics, and artificial intelligence (AI).[9] It has\nbeen referred to as "the most powerful company in the The Google logo used since 2015\nworld" by the BBC[10] and is one of the world\'s most\nvaluable brands.[11][12][13] Google\'s parent company,\nAlphabet Inc., is one of the five Big Tech companies\nalongside Amazon, Apple, Meta, and Microsoft.\n\nGoogle was founded on September 4, 1998, by\nAmerican computer scientists Larry Page and Sergey\nBrin. Together, they own about 14% of its publicly\nlisted shares and control 56% of its stockholder voting\npower through super-voting stock. The company went\npub

In [81]:
print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 5


### Chunking Documents

In [82]:
def split_documents(documents, chunk_size=800, chunk_overlap=0):
    """Split documents into smaller chunks."""
    # text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(documents)
    return chunks

In [83]:
chunks = split_documents(documents)

Created a chunk of size 949, which is longer than the specified 800
Created a chunk of size 922, which is longer than the specified 800
Created a chunk of size 892, which is longer than the specified 800
Created a chunk of size 825, which is longer than the specified 800
Created a chunk of size 921, which is longer than the specified 800
Created a chunk of size 830, which is longer than the specified 800
Created a chunk of size 1055, which is longer than the specified 800
Created a chunk of size 874, which is longer than the specified 800
Created a chunk of size 1436, which is longer than the specified 800
Created a chunk of size 924, which is longer than the specified 800
Created a chunk of size 815, which is longer than the specified 800
Created a chunk of size 1039, which is longer than the specified 800
Created a chunk of size 1078, which is longer than the specified 800
Created a chunk of size 1043, which is longer than the specified 800
Created a chunk of size 880, which is longe

In [84]:
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 1797


In [85]:
chunks[0].__dict__

{'id': None,
 'metadata': {'source': 'D:\\Projects\\Agentic-AI\\langchain-RAG\\docs\\Google.txt'},
 'page_content': '\ufeffGoogle\nGoogle LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC\nAmerican multinational corporation and technology\ncompany focusing on online advertising, search engine\ntechnology, cloud computing, computer software,\nquantum computing, e-commerce, consumer\nelectronics, and artificial intelligence (AI).[9] It has\nbeen referred to as "the most powerful company in the The Google logo used since 2015\nworld" by the BBC[10] and is one of the world\'s most\nvaluable brands.[11][12][13] Google\'s parent company,\nAlphabet Inc., is one of the five Big Tech companies\nalongside Amazon, Apple, Meta, and Microsoft.',
 'type': 'Document'}

### Embeding Documents

In [86]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    show_progress=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 55840.27it/s]


In [87]:
text = "Python installation guide for beginners"

vector = embeddings.embed_query(text)

print(len(vector))   # usually 1024 dimensions
print(vector[:5])    # first few values
print(type(vector))    # first few values

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.09it/s]

1024
[-0.03700488805770874, -0.009373921900987625, -0.06121456250548363, 0.027309292927384377, 0.013841482810676098]
<class 'list'>


In [88]:
docs = [
    "Python installation guide",
    "How to cook rice",
    "Machine learning basics"
]

doc_vectors = embeddings.embed_documents(docs)

print(len(doc_vectors))      # number of docs
print(len(doc_vectors[0]))   # embedding size
print(type(doc_vectors[0]))   # embedding size

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.41it/s]

3
1024
<class 'list'>


In [89]:
from langchain_community.utils.math import cosine_similarity

for doc in doc_vectors:
    score = cosine_similarity([vector], [doc])[0][0]
    print(score)

0.9584842125743365
0.4228729567460271
0.5204327585532278


### Saving Document in VectorStore

In [90]:
# embeddings Model: BAAI/bge-m3
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    show_progress=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 25148.33it/s]


In [ ]:
def create_vector_store(chunks, persist_directory="db/chroma_db"):
    """Create a vector store from the document chunks and embeddings."""
    # Create a Chroma vector store
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    return vector_store

In [92]:
vector_store = create_vector_store(chunks)

Batches: 100%|██████████| 57/57 [23:41<00:00, 24.94s/it] 


TypeError: object of type 'Chroma' has no len()

In [ ]:
    # print(f"Vector store saved at {persist_directory} with {len(vector_store)} documents.")
